# Comprobacion H&E

Comprobacion visual de la segmentacion H&E para `SB012`, `SB013`, `SB017`, `SB018`, `SB019` y `SB020`.

Carga las funciones de `9_funcion_preprocesado.ipynb`, igual que `10_guardar_imagenes_a_escala.ipynb`, y muestra original vs segmentada lado a lado.

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import nbformat
import numpy as np
from PIL import Image

plt.rcParams['figure.dpi'] = 120

BASE_DIR = Path(r'Registration\DeepLearning')
PREPROCESS_NOTEBOOK = BASE_DIR / '9_funcion_preprocesado.ipynb'
OUTPUT_DIR = BASE_DIR / 'Comprobaciones_segmentacion' / 'HE'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SUBJECTS = ['SB012', 'SB013', 'SB017', 'SB018', 'SB019', 'SB020']

print('Notebook de funciones:', PREPROCESS_NOTEBOOK)
print('Carpeta de salida:', OUTPUT_DIR)

## Cargar funciones del notebook 9

In [ ]:
def load_preprocess_definitions(notebook_path):
    nb = nbformat.read(str(notebook_path), as_version=4)
    skipped = []

    for idx, cell in enumerate(nb.cells):
        if cell.cell_type != 'code':
            continue

        source = cell.source
        skip_patterns = [
            'preprocess_output = preprocesar_he_hsi',
            'validation_results = {}',
        ]
        if any(pattern in source for pattern in skip_patterns):
            skipped.append(idx)
            continue

        exec(compile(source, f'{notebook_path.name}:cell_{idx}', 'exec'), globals())

    print(f'Celdas de ejemplo/validacion saltadas: {skipped}')


load_preprocess_definitions(PREPROCESS_NOTEBOOK)

## Funciones de visualizacion

In [ ]:

def _largest_component_he(mask, min_area=500):
    mask_u8 = (np.asarray(mask) > 0).astype(np.uint8)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask_u8, connectivity=8)
    if num_labels <= 1:
        return np.zeros(mask_u8.shape, dtype=bool)

    areas = stats[1:, cv2.CC_STAT_AREA]
    valid = np.where(areas >= min_area)[0]
    if len(valid) > 0:
        largest_label = 1 + valid[np.argmax(areas[valid])]
    else:
        largest_label = 1 + int(np.argmax(areas))
    return labels == largest_label


def extract_he_tissue_contour_image_stain_mask(
    slide_path,
    max_dim=1800,
    sat_thresh=10,
    val_thresh=252,
    chroma_thresh=4.0,
    min_area=500,
    open_kernel_size=3,
    close_kernel_size=65,
    grow_pixels=5,
    show_original=True,
    show_result=True,
    debug=False,
    return_mask=False,
    return_info=False,
):
    """Segmenta H&E usando semillas de tincion rosa/morada, evitando fondo blanco poco saturado."""
    slide_path = Path(slide_path)
    if not slide_path.exists():
        raise FileNotFoundError(f'No existe la H&E: {slide_path}')

    rgb_u8 = np.asarray(Image.open(slide_path).convert('RGB'), dtype=np.uint8)
    level_info = read_he_slidedat_info(slide_path)
    chosen_level = level_info['preview_level']
    level_w, level_h = rgb_u8.shape[1], rgb_u8.shape[0]

    hsv = cv2.cvtColor(rgb_u8, cv2.COLOR_RGB2HSV)
    sat = hsv[:, :, 1]
    val = hsv[:, :, 2]

    lab = cv2.cvtColor(rgb_u8, cv2.COLOR_RGB2LAB).astype(np.float32)
    lab_l = lab[:, :, 0]
    lab_a = lab[:, :, 1] - 128.0
    lab_b = lab[:, :, 2] - 128.0
    chroma = np.sqrt(lab_a * lab_a + lab_b * lab_b)

    mask_color = ((sat > sat_thresh) & (val < val_thresh)) | ((chroma > chroma_thresh) & (lab_l < 252))
    mask = mask_color.astype(np.uint8) * 255

    if open_kernel_size and open_kernel_size > 1:
        kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (open_kernel_size, open_kernel_size))
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel_open)

    if close_kernel_size and close_kernel_size > 1:
        kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (close_kernel_size, close_kernel_size))
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel_close)

    mask_clean = _largest_component_he(mask, min_area=min_area).astype(np.uint8) * 255

    contours, _ = cv2.findContours(mask_clean, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    mask_final = np.zeros_like(mask_clean, dtype=np.uint8)
    if contours:
        tissue_contour = max(contours, key=cv2.contourArea)
        cv2.drawContours(mask_final, [tissue_contour], -1, 255, thickness=cv2.FILLED)

    if grow_pixels > 0:
        kernel_grow = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * grow_pixels + 1, 2 * grow_pixels + 1))
        mask_final = cv2.dilate(mask_final, kernel_grow, iterations=1)

    tissue_only_rgb = rgb_u8.copy()
    tissue_only_rgb[mask_final == 0] = 0

    if debug:
        fig, axes = plt.subplots(2, 3, figsize=(14, 8), constrained_layout=True)
        panels = [
            (rgb_u8, f'H&E original | PIL preview level={chosen_level}', None),
            (sat, 'Saturacion HSV', 'gray'),
            (chroma, 'Croma LAB', 'gray'),
            (mask, 'Mascara inicial tincion', 'gray'),
            (mask_final, 'Mascara final por contorno', 'gray'),
            (tissue_only_rgb, 'Solo tejido', None),
        ]
        for ax, (img, title, cmap) in zip(axes.ravel(), panels):
            ax.imshow(img, cmap=cmap)
            ax.set_title(title)
            ax.axis('off')
        plt.show()
    elif show_original and show_result:
        fig, axes = plt.subplots(1, 2, figsize=(10, 6), constrained_layout=True)
        axes[0].imshow(rgb_u8)
        axes[0].set_title(f'H&E original | PIL preview level={chosen_level}')
        axes[0].axis('off')
        axes[1].imshow(tissue_only_rgb)
        axes[1].set_title('Solo tejido')
        axes[1].axis('off')
        plt.show()
    elif show_original:
        plt.figure(figsize=(6, 6))
        plt.imshow(rgb_u8)
        plt.title(f'H&E original | PIL preview level={chosen_level}')
        plt.axis('off')
        plt.show()
    elif show_result:
        plt.figure(figsize=(6, 6))
        plt.imshow(tissue_only_rgb)
        plt.title('Solo tejido')
        plt.axis('off')
        plt.show()

    outputs = [tissue_only_rgb]
    if return_mask:
        outputs.append(mask_final)
    if return_info:
        outputs.append({
            'method': 'he_stain_chroma_mask',
            'chosen_level': int(chosen_level),
            'read_dimensions': (int(level_w), int(level_h)),
            'sat_thresh': sat_thresh,
            'val_thresh': val_thresh,
            'chroma_thresh': chroma_thresh,
            'close_kernel_size': close_kernel_size,
            'grow_pixels': grow_pixels,
            'mask_area_px': int(np.count_nonzero(mask_final)),
            'reader': 'PIL',
            'slidedat_path': level_info['slidedat_path'],
        })
    return outputs[0] if len(outputs) == 1 else tuple(outputs)
def draw_mask_contour(rgb, mask, color=(0, 255, 255), thickness=3):
    out = np.asarray(rgb, dtype=np.uint8).copy()
    mask_u8 = (np.asarray(mask) > 0).astype(np.uint8) * 255
    contours, _ = cv2.findContours(mask_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(out, contours, -1, color, thickness)
    return out


def save_side_by_side(path, original_rgb, segmented_rgb, mask, title):
    segmented_with_contour = draw_mask_contour(segmented_rgb, mask)
    fig, axes = plt.subplots(1, 2, figsize=(10, 6), constrained_layout=True)
    axes[0].imshow(original_rgb)
    axes[0].set_title(f'{title} - original')
    axes[0].axis('off')
    axes[1].imshow(segmented_with_contour)
    axes[1].set_title(f'{title} - segmentada')
    axes[1].axis('off')
    fig.savefig(path, dpi=160, bbox_inches='tight')
    plt.show()
    plt.close(fig)

## Comprobacion por sujeto

In [ ]:
he_results = {}

for subject_id in SUBJECTS:
    print(f'=== {subject_id} ===')
    he_path = SPECIMENS[subject_id]['he_path']

    original_rgb, original_info = read_he_preview(he_path)
    segmented_rgb, mask, info = extract_he_tissue_contour_image_stain_mask(
        he_path,
        max_dim=1800,
        show_original=False,
        show_result=False,
        return_mask=True,
        return_info=True,
    )

    out_path = OUTPUT_DIR / f'{subject_id}_he_original_vs_segmentada.png'
    save_side_by_side(out_path, original_rgb, segmented_rgb, mask, f'{subject_id} H&E')

    he_results[subject_id] = {
        'he_path': he_path,
        'original_shape': original_rgb.shape,
        'segmented_shape': segmented_rgb.shape,
        'mask_area_px': int(np.count_nonzero(mask)),
        'segmentation_info': info,
        'plot_path': out_path,
    }

    print('Original:', original_rgb.shape)
    print('Segmentada:', segmented_rgb.shape, '| area mascara:', int(np.count_nonzero(mask)))
    print('Plot:', out_path)

he_results